# 🧠 Huấn Luyện AI Cờ Tướng (Xiangqi HalfKP NNUE)
### Kết nối Thư Viện Engine Nổi Tiếng (Pikafish, Fairy-Stockfish) & Huấn luyện Kỳ Phổ Đại Sư

Notebook này hướng dẫn và tự động hóa toàn bộ quy trình huấn luyện cho dự án Game Cờ Tướng:
1. **Tải & Cài đặt Engine ngoài**: Tải và thiết lập các Engine cờ tướng hàng đầu thế giới (**Pikafish** & **Fairy-Stockfish**) vào thư mục biệt lập `engines/` (được `.gitignore` bảo vệ).
2. **Huấn luyện Giám sát (Supervised Learning)**: Huấn luyện mạng NNUE HalfKP từ tập kỳ phổ mẫu danh thủ ở định dạng Float nguyên bản (`.f32`).
3. **Chưng cất Tri thức (Knowledge Distillation)**: Kế thừa mạng Float từ bước trước, dùng Pikafish làm "Sư phụ" đánh giá và gán nhãn tinh chỉnh thế cờ.
4. **Đấu tập Đối kháng (Sparring)**: Cho Bot với mô hình Float thi đấu trực tiếp với Engine ngoài để đo lường trình độ thực chiến.
5. **Lượng tử hóa Mô hình (Quantization)**: Gọi hàm lượng tử hóa riêng biệt để chuyển đổi mô hình Float f32 sang Int16/Int8 NNUE và lưu vào `cotuong_core/assets/default.nnue`.
6. **Kiểm thử & Đóng gói WASM**: Xác minh tính toàn vẹn và tự động biên dịch lại gói WebAssembly (`cotuong_core.wasm`) cho giao diện web.

> 💡 **Độ Chính Xác Tuyệt Đối (No Intermediate Quantization Loss)**: Toàn bộ quá trình huấn luyện và kế thừa giữa các bước đều sử dụng trọng số Float 32-bit (`checkpoints/model_latest.f32`), loại bỏ hoàn toàn sai số tích lũy do lượng tử hóa sớm. Chỉ bước cuối cùng mới thực hiện lượng tử hóa.


---
## 🛠️ Bước 1: Kiểm Tra Môi Trường & Khởi Tạo Thư Mục Làm Việc


In [11]:
# Tạo các thư mục lưu trữ riêng (đã được cấu hình trong .gitignore để không đẩy lên GitHub)
!mkdir -p engines checkpoints

import os
import sys
import subprocess

print("🔍 Kiểm tra môi trường hệ thống:")
!rustc --version
!cargo --version
!node --version
!npm --version

cpu_cores = os.cpu_count()
print(f"✅ Số nhân CPU khả dụng: {cpu_cores}")
print(f"📁 Thư mục Engines    : {os.path.abspath('engines')}")
print(f"📁 Thư mục Checkpoints: {os.path.abspath('checkpoints')}")


🔍 Kiểm tra môi trường hệ thống:
rustc 1.96.0 (ac68faa20 2026-05-25)
cargo 1.96.0 (30a34c682 2026-05-25)
v22.22.2
10.9.7
✅ Số nhân CPU khả dụng: 16
📁 Thư mục Engines    : /home/exblackhole/Desktop/GameCoTuong/engines
📁 Thư mục Checkpoints: /home/exblackhole/Desktop/GameCoTuong/checkpoints


---
## 🐟 Bước 2: Tải & Cài Đặt Engine Pikafish (UCCI / UCI Protocol)
**Pikafish** là engine cờ tướng mạnh nhất thế giới hiện nay (dựa trên nền tảng Stockfish). Ở bước này, chúng ta sẽ thiết lập binary Pikafish vào thư mục `engines/pikafish/`.


In [12]:
%%bash
set -e
ENGINES_DIR="$(pwd)/engines"
PIKAFISH_DIR="$ENGINES_DIR/pikafish"
mkdir -p "$PIKAFISH_DIR"

TARGET_BIN="$PIKAFISH_DIR/pikafish"

if [ -f "$TARGET_BIN" ]; then
    echo "✅ Pikafish đã có sẵn tại: $TARGET_BIN"
else
    echo "⏳ Đang chuẩn bị Engine Pikafish..."
    cd "$PIKAFISH_DIR"
    
    DOWNLOAD_SUCCESS=0
    RELEASE_URL=$(curl -s https://api.github.com/repos/official-pikafish/Pikafish/releases/latest | grep "browser_download_url.*pikafish-.*x86-64.*\.zip" | head -n 1 | cut -d '"' -f 4 || true)
    
    if [ -n "$RELEASE_URL" ]; then
        echo "⬇️ Đang tải bản phát hành từ: $RELEASE_URL"
        curl -L -o pikafish_release.zip "$RELEASE_URL"
        unzip -o pikafish_release.zip
        rm -f pikafish_release.zip
        FOUND_BIN=$(find . -maxdepth 2 -type f -name "pikafish*" ! -name "*.zip" ! -name "*.nnue" | head -n 1)
        if [ -n "$FOUND_BIN" ]; then
            chmod +x "$FOUND_BIN"
            cp "$FOUND_BIN" "$TARGET_BIN"
            DOWNLOAD_SUCCESS=1
        fi
    fi
    
    if [ "$DOWNLOAD_SUCCESS" -eq 0 ]; then
        echo "🛠️ Đang biên dịch Pikafish từ nguồn..."
        rm -rf /tmp/pikafish_src
        git clone --depth 1 https://github.com/official-pikafish/Pikafish.git /tmp/pikafish_src
        make -C /tmp/pikafish_src/src -j$(nproc) build ARCH=x86-64-modern
        cp /tmp/pikafish_src/src/pikafish "$TARGET_BIN"
        chmod +x "$TARGET_BIN"
        rm -rf /tmp/pikafish_src
    fi
fi

echo "🧪 Kiểm thử giao thức UCCI với Pikafish..."
echo -e "ucci\nquit" | "$TARGET_BIN" | head -n 6
echo "✅ Pikafish đã sẵn sàng tại: $TARGET_BIN"


✅ Pikafish đã có sẵn tại: /home/exblackhole/Desktop/GameCoTuong/engines/pikafish/pikafish
🧪 Kiểm thử giao thức UCCI với Pikafish...
Pikafish dev-20261029-fec5d430 by the Pikafish developers (see AUTHORS file)
Unknown command: 'ucci'. Type help for more information.
✅ Pikafish đã sẵn sàng tại: /home/exblackhole/Desktop/GameCoTuong/engines/pikafish/pikafish


---
## 🧚 Bước 3: Tải & Cài Đặt Engine Fairy-Stockfish (UCI Protocol)
**Fairy-Stockfish** là engine nguồn mở chuyên về các biến thể cờ tướng phương Đông và cờ vua phương Tây, hỗ trợ giao thức UCI chuẩn mực với tùy chọn `setoption name UCI_Variant value xiangqi`.


In [13]:
%%bash
set -e
ENGINES_DIR="$(pwd)/engines"
FAIRY_DIR="$ENGINES_DIR/fairy-stockfish"
mkdir -p "$FAIRY_DIR"

TARGET_BIN="$FAIRY_DIR/fairy-stockfish"

if [ -f "$TARGET_BIN" ]; then
    echo "✅ Fairy-Stockfish đã có sẵn tại: $TARGET_BIN"
else
    echo "⏳ Đang chuẩn bị Engine Fairy-Stockfish..."
    cd "$FAIRY_DIR"
    
    DOWNLOAD_SUCCESS=0
    RELEASE_URL=$(curl -s https://api.github.com/repos/fairy-stockfish/Fairy-Stockfish/releases/latest | grep "browser_download_url.*largeboard_x86-64-modern" | head -n 1 | cut -d '"' -f 4 || true)
    
    if [ -n "$RELEASE_URL" ]; then
        echo "⬇️ Đang tải từ: $RELEASE_URL"
        curl -L -o "$TARGET_BIN" "$RELEASE_URL"
        chmod +x "$TARGET_BIN"
        DOWNLOAD_SUCCESS=1
    fi
    
    if [ "$DOWNLOAD_SUCCESS" -eq 0 ]; then
        echo "🛠️ Đang biên dịch Fairy-Stockfish từ nguồn..."
        rm -rf /tmp/fairy_src
        git clone --depth 1 https://github.com/fairy-stockfish/Fairy-Stockfish.git /tmp/fairy_src
        make -C /tmp/fairy_src/src -j$(nproc) build ARCH=x86-64-modern
        cp /tmp/fairy_src/src/fairy-stockfish "$TARGET_BIN"
        chmod +x "$TARGET_BIN"
        rm -rf /tmp/fairy_src
    fi
fi

echo "🧪 Kiểm thử giao thức UCI với Fairy-Stockfish..."
echo -e "uci\nsetoption name UCI_Variant value xiangqi\nisready\nquit" | "$TARGET_BIN" | grep -E "id name|readyok"
echo "✅ Fairy-Stockfish đã sẵn sàng tại: $TARGET_BIN"


✅ Fairy-Stockfish đã có sẵn tại: /home/exblackhole/Desktop/GameCoTuong/engines/fairy-stockfish/fairy-stockfish
🧪 Kiểm thử giao thức UCI với Fairy-Stockfish...
id name Fairy-Stockfish 14 LB
readyok
✅ Fairy-Stockfish đã sẵn sàng tại: /home/exblackhole/Desktop/GameCoTuong/engines/fairy-stockfish/fairy-stockfish


---
## ⚔️ Bước 3.5: Sinh Dữ Liệu Thi Đấu Tự Động giữa Pikafish và Fairy-Stockfish (Arena Data Generation)
Khi tập dữ liệu mẫu ban đầu quá ít, ta cho 2 đại sư phụ AI hàng đầu thế giới **Pikafish** và **Fairy-Stockfish** tự động đấu cờ với nhau ở độ sâu cao (**depth 15**) qua các biến thể khai cuộc phong phú để sinh hàng nghìn ván cờ và thế cờ chất lượng cao (Master FEN Evaluations) lưu vào `cotuong_trainer/data/sample_master_games.txt` phục vụ huấn luyện:


In [14]:
# Cho Pikafish và Fairy-Stockfish thi đấu đối kháng để sinh dữ liệu mẫu
!cargo run --release -p cotuong_trainer -- \
    --mode gendata \
    --engine1 engines/pikafish/pikafish \
    --engine2 engines/fairy-stockfish/fairy-stockfish \
    --games 20 \
    --depth 15 \
    --dataset cotuong_trainer/data/sample_master_games.txt


   Compiling cotuong_core v0.1.0 (/home/exblackhole/Desktop/GameCoTuong/cotuong_core)
   Compiling cotuong_trainer v0.1.0 (/home/exblackhole/Desktop/GameCoTuong/cotuong_trainer)
    Finished `release` profile [optimized] target(s) in 7.98sr(bin)        
     Running `target/release/cotuong_trainer --mode gendata --engine1 engines/pikafish/pikafish --engine2 engines/fairy-stockfish/fairy-stockfish --games 20 --depth 6 --dataset cotuong_trainer/data/sample_master_games.txt`
 🧠 Xiangqi AI Trainer (Master Dataset & External UCCI/UCI Engine Integration)  
Cấu hình:
  • Chế độ (Mode)            : gendata
  • Số Epochs                : 5
  • Tốc độ học (LR)          : 0.05
  • Đường dẫn trọng số đích : checkpoints/model_latest.f32
  • Kế thừa trọng số         : Tự động từ checkpoints/model_latest.f32 (nếu có)
  • Thư mục Checkpoint       : checkpoints

▶ Khởi tạo Đấu Trường Sinh Dữ Liệu (Arena Engine Match):
   • Engine 1: engines/pikafish/pikafish (Uci)
   • Engine 2: engines/fairy-stockfish

---
## 📖 Bước 4: Huấn Luyện Giám Sát từ Kỳ Phổ Mẫu (Supervised Learning)
Huấn luyện mạng NNUE học ngay từ các thế cờ danh thủ trong `cotuong_trainer/data/sample_master_games.txt`.

- **Định dạng**: Trọng số lưu ở dạng Float 32-bit (`checkpoints/model_latest.f32`) chưa lượng tử hóa để đảm bảo độ chính xác tối đa.
- **Kế thừa**: Tự động nạp lại trọng số Float đã có trước đó (hoặc dùng `--resume <file>`). Thêm `--scratch` nếu muốn huấn luyện lại từ đầu hoàn toàn.
- **Checkpoints**: Lưu từng thế hệ vào `checkpoints/dataset_epoch_{N}.f32`.


In [ ]:
# Chạy huấn luyện Supervised Learning 5 Epochs (Lưu trọng số dạng Float f32 chính xác cao)
!cargo run --release -p cotuong_trainer -- \
    --mode dataset \
    --epochs 20 \
    --lr 0.01 \
    --eval-games 10 \
    --checkpoints checkpoints \
    --output checkpoints/model_latest.f32


   Compiling cotuong_trainer v0.1.0 (/home/exblackhole/Desktop/GameCoTuong/cotuong_trainer)
    Finished `release` profile [optimized] target(s) in 6.35sr(bin)        
     Running `target/release/cotuong_trainer --mode dataset --epochs 20 --lr 0.05 --eval-games 10 --checkpoints checkpoints --output checkpoints/model_latest.f32`
 🧠 Xiangqi AI Trainer (Master Dataset & External UCCI/UCI Engine Integration)  
Cấu hình:
  • Chế độ (Mode)            : dataset
  • Số Epochs                : 20
  • Tốc độ học (LR)          : 0.05
  • Đường dẫn trọng số đích : checkpoints/model_latest.f32
  • Kế thừa trọng số         : Tự động từ checkpoints/model_latest.f32 (nếu có)
  • Thư mục Checkpoint       : checkpoints
  • Tập dữ liệu kỳ phổ mẫu   : cotuong_trainer/data/sample_master_games.txt
  • Số ván Đấu Đánh Giá      : 10

▶ Nạp dữ liệu kỳ phổ mẫu từ: cotuong_trainer/data/sample_master_games.txt ...
  • Đã nạp thành công 5117 thế cờ mẫu!
♻️  Tự động nạp trọng số chưa lượng tử hóa từ: checkpoints/m

---
## 👨‍🏫 Bước 5: Chưng Cất Tri Thức cùng Sư Phụ Pikafish (Knowledge Distillation)
Kế thừa trọng số Float f32 từ Bước 4 (`checkpoints/model_latest.f32`), dùng Pikafish làm "Sư phụ" để gán nhãn điểm số (`search_eval` ở độ sâu 10–14) và tiếp tục tối ưu hóa mạng ở độ chính xác Float f32. Chế độ này chỉ sinh checkpoint duy nhất ở epoch cuối cùng với tên file đặt theo thời điểm huấn luyện xong: `teacher_YYYYMMDD_HHMMSS.f32`.


In [ ]:
# Chạy chưng cất tri thức với Engine sư phụ Pikafish (Kế thừa Float f32 model)
!cargo run --release -p cotuong_trainer -- \
    --mode teacher \
    --engine engines/pikafish/pikafish \
    --proto ucci \
    --depth 15 \
    --epochs 100 \
    --lr 0.01 \
    --checkpoints checkpoints \
    --output checkpoints/model_latest.f32


    Finished `release` profile [optimized] target(s) in 0.02s
     Running `target/release/cotuong_trainer --mode teacher --engine engines/pikafish/pikafish --proto ucci --depth 15 --epochs 10 --lr 0.01 --checkpoints checkpoints --output checkpoints/model_latest.f32`
 🧠 Xiangqi AI Trainer (Master Dataset & External UCCI/UCI Engine Integration)  
Cấu hình:
  • Chế độ (Mode)            : teacher
  • Số Epochs                : 10
  • Tốc độ học (LR)          : 0.01
  • Đường dẫn trọng số đích : checkpoints/model_latest.f32
  • Kế thừa trọng số         : Tự động từ checkpoints/model_latest.f32 (nếu có)
  • Thư mục Checkpoint       : checkpoints
  • Engine Sư Phụ            : engines/pikafish/pikafish (Ucci)
  • Độ sâu phân tích         : 15
  • Tập thế cờ gán nhãn      : cotuong_trainer/data/sample_master_games.txt

▶ Khởi động Engine Sư Phụ ("engines/pikafish/pikafish", protocol: Ucci) ...
✅ Đã kết nối thành công với Engine sư phụ qua giao thức chuẩn!
▶ Nạp tập thế cờ cần gán nhãn từ: cot

---
## 🥊 Bước 6: Đấu Tập Đối Kháng Trực Tiếp với Engine Ngoài (Sparring Match)
Nạp mô hình Float đã huấn luyện và chưng cất (`checkpoints/model_latest.f32`) để thi đấu đối kháng trực tiếp luân phiên Đỏ / Đen với Pikafish hoặc Fairy-Stockfish. Trọng số mô hình được bảo toàn làm chuẩn benchmark.


In [26]:
# Đấu tập 10 ván với Engine ngoài (Pikafish / Fairy-Stockfish) ở độ sâu 15
!cargo run --release -p cotuong_trainer -- \
    --mode sparring \
    --engine engines/pikafish/pikafish \
    --proto ucci \
    --games 10 \
    --depth 15 \
    --resume checkpoints/model_latest.f32


    Finished `release` profile [optimized] target(s) in 0.02s
     Running `target/release/cotuong_trainer --mode sparring --engine engines/pikafish/pikafish --proto ucci --games 10 --depth 15 --resume checkpoints/model_latest.f32`
 🧠 Xiangqi AI Trainer (Master Dataset & External UCCI/UCI Engine Integration)  
Cấu hình:
  • Chế độ (Mode)            : sparring
  • Số Epochs                : 5
  • Tốc độ học (LR)          : 0.05
  • Đường dẫn trọng số đích : checkpoints/model_latest.f32
  • Nạp trọng số khởi điểm  : checkpoints/model_latest.f32
  • Thư mục Checkpoint       : checkpoints
  • Engine Đối thủ           : engines/pikafish/pikafish (Ucci)
  • Số ván đấu               : 10
  • Độ sâu đấu tập           : 15

▶ Khởi tạo trận đấu tập (Sparring) với Engine: engines/pikafish/pikafish ...
✅ Đã sẵn sàng thi đấu 10 ván đối kháng!
♻️  Đã nạp thành công trọng số từ: checkpoints/model_latest.f32 để tiếp tục huấn luyện!
  ▶ Ván 1/10: Bot cầm ĐỎ
  ▶ Ván 2/10: Bot cầm ĐEN
  ▶ Ván 3/10: Bot c

---
## 🎯 Bước 7: Lượng Tử Hóa Mô Hình (Quantization: Float f32 ➔ Int16/Int8 NNUE)
Đây là bước duy nhất thực hiện lượng tử hóa trong toàn bộ quy trình:
- **Đầu vào**: Mô hình Float chính xác cao sau khi huấn luyện xong (`checkpoints/model_latest.f32`).
- **Hàm thực hiện**: Chế độ `--mode quantize` riêng biệt của `cotuong_trainer`.
- **Đầu ra**: Tệp nhị phân Int16/Int8 được nén và lượng tử hóa, lưu trực tiếp vào `cotuong_core/assets/default.nnue`.


In [27]:
# Lượng tử hóa mô hình Float f32 sang Int16/Int8 NNUE và lưu vào cotuong_core/assets/default.nnue
!cargo run --release -p cotuong_trainer -- \
    --mode quantize \
    --input checkpoints/model_latest.f32 \
    --output cotuong_core/assets/default.nnue


    Finished `release` profile [optimized] target(s) in 0.02s
     Running `target/release/cotuong_trainer --mode quantize --input checkpoints/model_latest.f32 --output cotuong_core/assets/default.nnue`
 🧠 Xiangqi AI Trainer (Master Dataset & External UCCI/UCI Engine Integration)  
Cấu hình:
  • Chế độ (Mode)            : quantize
  • File Float nguồn         : checkpoints/model_latest.f32
  • File NNUE đích           : cotuong_core/assets/default.nnue

▶ Bắt đầu lượng tử hóa mô hình Float NNUE (f32) -> Quantized NNUE (i16/i8)...
  • File nguồn (Float NNUE)    : checkpoints/model_latest.f32
  • File đích (Quantized NNUE) : cotuong_core/assets/default.nnue
✅ Đã lượng tử hóa và lưu thành công sang: cotuong_core/assets/default.nnue
  • Dung lượng file lượng tử hóa : 4127.0 KB


---
## 🗄️ Bước 8: Quản Lý Checkpoints, Kiểm Thử & Đóng Gói WebAssembly (WASM)
1. Liệt kê các checkpoint Float (`.f32`) và kiểm tra dung lượng tệp lượng tử hóa `default.nnue`.
2. Kiểm thử tính hợp lệ của trọng số nhúng và Accumulator trong `cotuong_core`.
3. Tự động biên dịch lại gói WebAssembly (`cotuong_core.wasm`) sẵn sàng cho giao diện web.


In [28]:
import glob
import os

print("📦 Danh sách các Checkpoints Float đã lưu trong checkpoints/:")
float_checkpoints = sorted(glob.glob("checkpoints/*.f32"))
if float_checkpoints:
    for cp in float_checkpoints:
        size_mb = os.path.getsize(cp) / (1024.0 * 1024.0)
        print(f"  • {cp} ({size_mb:.2f} MB)")
else:
    print("  (Chưa có file checkpoint float nào, hãy chạy bước huấn luyện ở trên)")

print("\n▶ Trọng số chính đã lượng tử hóa:")
default_nnue = "cotuong_core/assets/default.nnue"
if os.path.exists(default_nnue):
    print(f"  • {default_nnue} ({os.path.getsize(default_nnue) / 1024.0:.1f} KB)")


📦 Danh sách các Checkpoints Float đã lưu trong checkpoints/:
  • checkpoints/dataset_epoch_1.f32 (8.09 MB)
  • checkpoints/dataset_epoch_10.f32 (8.09 MB)
  • checkpoints/dataset_epoch_11.f32 (8.09 MB)
  • checkpoints/dataset_epoch_12.f32 (8.09 MB)
  • checkpoints/dataset_epoch_13.f32 (8.09 MB)
  • checkpoints/dataset_epoch_14.f32 (8.09 MB)
  • checkpoints/dataset_epoch_15.f32 (8.09 MB)
  • checkpoints/dataset_epoch_16.f32 (8.09 MB)
  • checkpoints/dataset_epoch_17.f32 (8.09 MB)
  • checkpoints/dataset_epoch_18.f32 (8.09 MB)
  • checkpoints/dataset_epoch_2.f32 (8.09 MB)
  • checkpoints/dataset_epoch_20.f32 (8.09 MB)
  • checkpoints/dataset_epoch_3.f32 (8.09 MB)
  • checkpoints/dataset_epoch_4.f32 (8.09 MB)
  • checkpoints/dataset_epoch_5.f32 (8.09 MB)
  • checkpoints/dataset_epoch_6.f32 (8.09 MB)
  • checkpoints/dataset_epoch_7.f32 (8.09 MB)
  • checkpoints/dataset_epoch_8.f32 (8.09 MB)
  • checkpoints/dataset_epoch_9.f32 (8.09 MB)
  • checkpoints/model_latest.f32 (8.09 MB)
  • checkpoi

In [29]:
# Kiểm thử tính hợp lệ và Accumulator của model NNUE vừa lượng tử hóa
!cargo test -p cotuong_core -- test_embedded_weights_validity test_accumulator_incremental_vs_refresh

# Biên dịch lại gói WebAssembly (WASM) với trọng số mới
!npm run build:wasm
print("\n🎉 Hoàn tất đóng gói! Bạn có thể chạy 'npm run dev' trên terminal để trải nghiệm AI trên trình duyệt.")


   Compiling cotuong_core v0.1.0 (/home/exblackhole/Desktop/GameCoTuong/cotuong_core)
    Finished `test` profile [unoptimized + debuginfo] target(s) in 0.52s   
     Running unittests src/lib.rs (target/debug/deps/cotuong_core-132c201bcd89cba7)

running 2 tests
test engine::nnue::tests::test_embedded_weights_validity ... ok
test engine::nnue::tests::test_accumulator_incremental_vs_refresh ... ok

test result: ok. 2 passed; 0 failed; 0 ignored; 0 measured; 26 filtered out; finished in 0.06s

     Running tests/mate_solving.rs (target/debug/deps/mate_solving-d1aa54a08addb97b)

running 0 tests

test result: ok. 0 passed; 0 failed; 0 ignored; 0 measured; 4 filtered out; finished in 0.00s

   Doc-tests cotuong_core

running 0 tests

test result: ok. 0 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.00s


> game-co-tuong@0.1.0 build:wasm
> cargo build -p cotuong_core --target wasm32-unknown-unknown --release && wasm-bindgen --target web --out-dir src/wasm target/wasm3